#Imports

In [1]:
!pip install woodelf_explainer==0.2.14

In [2]:
import xgboost as xgb
import pandas as pd
import numpy as np
from typing import Union, Dict, Optional, Tuple, Set, List
from math import factorial
import time
from copy import copy
from tqdm import tqdm
from collections import defaultdict
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
import scipy
import shap

import lightgbm as lgb

# For GPU execution
# import cupy as cp

In [3]:
from woodelf.parse_models import load_decision_tree_ensemble_model
from woodelf.cube_metric import CubeMetric, ShapleyInteractionValues, ShapleyValues

import woodelf


In [4]:
# Useful if you run this on google colab and downloaded the data into your drive.
# If you run the notebook in other environment remove these lines and change the 'pd.read_csv()' function in this notebook to read from
# where you saved you data
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# import woodelfHD from Python file in the drive
!cp /content/drive/MyDrive/ShapResearch/HighDepth/AAAI27/woodelfhd.py /content/

import woodelfhd
from woodelfhd import woodelf_for_high_depth

## Environment Note

Use the CPU enviroment with High-RAM option enabled. It is available for subscribed members for Google Colab (toggle the High-RAM option in Runtime->Change runtime type button).

If you only bought computation units, but not a subscription, you can still run the full notebook using the v2-8-TPU machine. This machine have more then enough RAM and it is pretty cheap. Its CPU is a bit slower though, so you will get a slower runtimes.

Final note: preformance in Colab depends on the machine allocated (the CPU option can allocate several types of machines) and on load balancing inside Colab (when more users use the system running times might be slower). So the running times of our algorithms and the shap python package can very. The difference between our approach and the current state-of-the-art is big enough to be noticed, but the excat running times can be slightly different from the ones stated our paper.

# HIGGs Data Loading and Model training

In [6]:
# This loading process was done in the Preprocess_Data notebook
# URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00280/HIGGS.csv.gz"
# # UCI file format: first column is label, then 28 float features.
# cols = ["label"] + [f"f{i}" for i in range(28)]
# df = pd.read_csv(URL, header=None, names=cols)

df = pd.read_parquet("drive/MyDrive/ShapResearch/HighDepth/AAAI27/data/higgs_data.parquet")

X = df.drop(columns="label")
y = df["label"].astype(int)

higgs_train, higgs_test, higgs_y_train, higgs_y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [7]:
LIGHTGBM_PARAMS = {
    "boosting_type": "gbdt",
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.1,

    # Allow high depth and enough leaves to reach this possible depth
    "num_leaves": 2024,
    "max_depth": 10,
    "min_data_in_leaf": 500,        # Does provide some regulation

    # Sampling (stability + reduces overfit)
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,

    # Practical
    "verbosity": -1,
    "seed": 42,
    "force_col_wise": True,            # often faster/safer for wide data
}


def print_model_stat(model, features):
    mobj = load_decision_tree_ensemble_model(model, features)
    depths = {}
    for tree in mobj.trees:
        for leaf, path in tree.get_all_leaves_with_paths():
            depth = len(path)
            if depth not in depths:
                depths[depth] = 0
            depths[depth] += 1

    for depth in sorted(list(depths.keys())):
        print(f"\tPaths of depth {depth}: {depths.get(depth, 0)} paths")

def lightgbm_model(X_train, y_train, params, num_rounds=100):
    train_set = lgb.Dataset(X_train, label=y_train, free_raw_data=False)
    return lgb.train(
        params=params,
        train_set=train_set,
        num_boost_round=num_rounds
    )


def different_depth_lightgbm_models(trainset, y, params, num_rounds, depths):
    models = {}
    for depth in depths:
        new_params = params.copy()
        new_params['max_depth'] = depth
        models[depth] = lightgbm_model(trainset, y, new_params, num_rounds=num_rounds)
        print("\n\n")
        print(f"Trained on depth {depth}")
        print_model_stat(models[depth], list(trainset.columns))
    return models

In [8]:
gbm_100trees = different_depth_lightgbm_models(higgs_train, higgs_y_train, LIGHTGBM_PARAMS, num_rounds=100, depths=[6,9])




Trained on depth 6
	Paths of depth 3: 1 paths
	Paths of depth 4: 22 paths
	Paths of depth 5: 83 paths
	Paths of depth 6: 6138 paths



Trained on depth 9
	Paths of depth 3: 9 paths
	Paths of depth 4: 37 paths
	Paths of depth 5: 130 paths
	Paths of depth 6: 382 paths
	Paths of depth 7: 1109 paths
	Paths of depth 8: 2848 paths
	Paths of depth 9: 34172 paths


In [9]:
gbm_10trees = different_depth_lightgbm_models(higgs_train, higgs_y_train, LIGHTGBM_PARAMS, num_rounds=10, depths=[6, 9, 12,15])




Trained on depth 6
	Paths of depth 6: 640 paths



Trained on depth 9
	Paths of depth 6: 4 paths
	Paths of depth 7: 31 paths
	Paths of depth 8: 183 paths
	Paths of depth 9: 4598 paths



Trained on depth 12
	Paths of depth 6: 11 paths
	Paths of depth 7: 54 paths
	Paths of depth 8: 253 paths
	Paths of depth 9: 833 paths
	Paths of depth 10: 1725 paths
	Paths of depth 11: 3552 paths
	Paths of depth 12: 13812 paths



Trained on depth 15
	Paths of depth 5: 2 paths
	Paths of depth 6: 24 paths
	Paths of depth 7: 125 paths
	Paths of depth 8: 420 paths
	Paths of depth 9: 1023 paths
	Paths of depth 10: 1988 paths
	Paths of depth 11: 2919 paths
	Paths of depth 12: 3465 paths
	Paths of depth 13: 3593 paths
	Paths of depth 14: 3019 paths
	Paths of depth 15: 3662 paths


In [10]:
gbm_1trees = different_depth_lightgbm_models(higgs_train, higgs_y_train, LIGHTGBM_PARAMS, num_rounds=1, depths=[6, 9, 12,15,18, 21])




Trained on depth 6
	Paths of depth 6: 64 paths



Trained on depth 9
	Paths of depth 7: 3 paths
	Paths of depth 8: 25 paths
	Paths of depth 9: 450 paths



Trained on depth 12
	Paths of depth 7: 9 paths
	Paths of depth 8: 23 paths
	Paths of depth 9: 81 paths
	Paths of depth 10: 174 paths
	Paths of depth 11: 359 paths
	Paths of depth 12: 1378 paths



Trained on depth 15
	Paths of depth 6: 4 paths
	Paths of depth 7: 8 paths
	Paths of depth 8: 45 paths
	Paths of depth 9: 92 paths
	Paths of depth 10: 212 paths
	Paths of depth 11: 313 paths
	Paths of depth 12: 353 paths
	Paths of depth 13: 369 paths
	Paths of depth 14: 304 paths
	Paths of depth 15: 324 paths



Trained on depth 18
	Paths of depth 6: 4 paths
	Paths of depth 7: 12 paths
	Paths of depth 8: 41 paths
	Paths of depth 9: 95 paths
	Paths of depth 10: 213 paths
	Paths of depth 11: 296 paths
	Paths of depth 12: 328 paths
	Paths of depth 13: 332 paths
	Paths of depth 14: 270 paths
	Paths of depth 15: 191 paths
	Paths of depth 16: 

In [11]:
def high_depth_woodelf_on_models_dict(
        models, consumer_data: pd.DataFrame, background_data: pd.DataFrame, metric: CubeMetric,global_importance: bool = False, GPU=False
    ):
    running_times = {}
    for key, model in models.items():
        start_time = time.time()
        woodelf_for_high_depth(model, consumer_data, background_data, metric=metric, global_importance=global_importance)
        running_time = time.time() - start_time
        running_times[key] = running_time
        print(f"On Depth {key} Took: {running_time}")
    return running_times

In [12]:
def simple_woodelf_on_models_dict(
        models, consumer_data: pd.DataFrame, background_data: pd.DataFrame, metric: CubeMetric, global_importance: bool = False, GPU=False,
    ):
    running_times = {}
    for key, model in models.items():
        start_time = time.time()
        woodelf.simple_woodelf.calculate_background_metric(model, consumer_data, background_data, metric=metric, global_importance=global_importance)
        running_time = time.time() - start_time
        running_times[key] = running_time
        print(f"On Depth {key} Took: {running_time}")
    return running_times

In [13]:
def shap_on_models_dict(models, consumer_data: pd.DataFrame, background_data: pd.DataFrame):
    running_times = {}
    for key, model in models.items():
        start_time = time.time()
        explainer = shap.TreeExplainer(model, background_data, feature_perturbation='interventional')
        simple_shap_values = explainer.shap_values(consumer_data, check_additivity=False)
        running_time = time.time() - start_time
        running_times[key] = running_time
        print(f"On Depth {key} Took: {running_time}")
    return running_times

# Background SHAP

In [14]:
woodelf_running_times = high_depth_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_10trees[9], 12: gbm_1trees[12], 15: gbm_1trees[15], 18: gbm_1trees[18], 21: gbm_1trees[21]},
    consumer_data=higgs_test, background_data=higgs_train, metric=ShapleyValues(),
    global_importance = False, GPU=False
)
print("Background SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [08:08<00:00,  4.89s/it]


M time: 0.0 sec, s time: 1.0 sec (f prepare time: 0.35602498054504395)
On Depth 6 Took: 488.9519281387329


Preprocessing the trees and computing SHAP: 100%|██████████| 10/10 [08:42<00:00, 52.23s/it]


M time: 0.01 sec, s time: 0.95 sec (f prepare time: 0.3258347511291504)
On Depth 9 Took: 522.6555953025818


Preprocessing the trees and computing SHAP: 100%|██████████| 1/1 [04:06<00:00, 246.43s/it]


...

M time: 1.29 sec, s time: 0.62 sec (f prepare time: 0.1929769515991211)
On Depth 15 Took: 308.2898106575012


Preprocessing the trees and computing SHAP: 100%|██████████| 1/1 [06:10<00:00, 370.89s/it]


M time: 12.38 sec, s time: 0.66 sec (f prepare time: 0.19757771492004395)
On Depth 18 Took: 383.35366439819336


Preprocessing the trees and computing SHAP: 100%|██████████| 1/1 [06:40<00:00, 400.80s/it]

M time: 108.75 sec, s time: 0.71 sec (f prepare time: 0.2113630771636963)
On Depth 21 Took: 509.6343801021576
Background SHAP: 488.9519281387329 & 522.6555953025818 & 246.57291293144226 & 308.2898106575012 & 383.35366439819336 & 509.6343801021576


In [ ]:
woodelf_running_times = simple_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_10trees[9]}, # Depth 12 crashes due to RAM (see Crash Due to RAM sec below)
    consumer_data=higgs_test, background_data=higgs_train, metric=ShapleyValues(),
    global_importance = False, GPU=False
)
print("Background SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees: 100%|██████████| 100/100 [02:15<00:00,  1.36s/it]


cache misses: 164, cache used: 6080, M computation time: 0.47 sec, s computation time: 0.2 sec


Computing the values: 100%|██████████| 100/100 [01:24<00:00,  1.18it/s]


On Depth 6 Took: 220.58854746818542


Preprocessing the trees: 100%|██████████| 10/10 [04:39<00:00, 27.96s/it]


cache misses: 1352, cache used: 3464, M computation time: 146.99 sec, s computation time: 0.62 sec


Computing the values: 100%|██████████| 10/10 [01:32<00:00,  9.29s/it]


On Depth 9 Took: 372.64721941947937


Preprocessing the trees:   0%|          | 0/1 [00:00<?, ?it/s]

In [14]:
train_sample = higgs_train.sample(10, random_state=42)

shap_running_times = shap_on_models_dict(
    gbm_1trees,
    consumer_data=higgs_test, background_data=train_sample,
)
print("Background SHAP: " + " & ".join([str(shap_running_times[d]) for d in sorted(list(shap_running_times.keys()))]))

100%|===================| 2195949/2200000 [02:21<00:00]       

On Depth 6 Took: 141.93712711334229


100%|===================| 2197712/2200000 [06:44<00:00]       

On Depth 9 Took: 405.1875720024109


100%|===================| 2198780/2200000 [14:25<00:00]       

...

On Depth 15 Took: 1036.6533567905426


100%|===================| 2199407/2200000 [17:44<00:00]       

On Depth 18 Took: 1064.1535158157349


100%|===================| 2198751/2200000 [17:43<00:00]       

On Depth 21 Took: 1064.3194637298584
Background SHAP: 141.93712711334229 & 405.1875720024109 & 866.0597631931305 & 1036.6533567905426 & 1064.1535158157349 & 1064.3194637298584


Results on 10 trees

100%|===================| 2199296/2200000 [24:40<00:00]       On Depth 6 Took: 1480.4687597751617
  4%|=                   | 77295/2200000 [02:32<69:34]       


# Background SHAP IV

In [15]:
woodelf_running_times = high_depth_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_10trees[9], 12: gbm_1trees[12], 15: gbm_1trees[15], 18: gbm_1trees[18]}, # depth 21 crashes due to RAM (see the crashes on the fraud data)
    consumer_data=higgs_test.sample(10_000, random_state=42), background_data=higgs_train, metric=ShapleyInteractionValues(),
    global_importance = False, GPU=False
)
print("Background SHAP IV: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [03:55<00:00,  2.35s/it]


M time: 0.0 sec, s time: 1.06 sec (f prepare time: 0.33611249923706055)
On Depth 6 Took: 235.40930771827698


Preprocessing the trees and computing SHAP: 100%|██████████| 10/10 [04:45<00:00, 28.56s/it]


M time: 0.02 sec, s time: 1.08 sec (f prepare time: 0.31818270683288574)
On Depth 9 Took: 285.85532212257385


Preprocessing the trees and computing SHAP: 100%|██████████| 1/1 [02:08<00:00, 128.41s/it]


M time: 0.78 sec, s time: 0.63 sec (f prepare time: 0.15541839599609375)
On Depth 12 Took: 129.2672085762024


Preprocessing the trees and computing SHAP: 100%|██████████| 1/1 [02:26<00:00, 146.19s/it]


M time: 5.44 sec, s time: 0.84 sec (f prepare time: 0.18044281005859375)
On Depth 15 Took: 151.71453285217285


Preprocessing the trees and computing SHAP: 100%|██████████| 1/1 [03:01<00:00, 181.57s/it]

M time: 61.65 sec, s time: 0.99 sec (f prepare time: 0.18679285049438477)
On Depth 18 Took: 243.308185338974
Background SHAP IV: 235.40930771827698 & 285.85532212257385 & 129.2672085762024 & 151.71453285217285 & 243.308185338974


In [16]:
woodelf_running_times = simple_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_10trees[9]}, # depth 12 crashes due to RAM (see Crash Due to RAM sec below)
    consumer_data=higgs_test.sample(10_000, random_state=42), background_data=higgs_train, metric=ShapleyInteractionValues(),
    global_importance = False, GPU=False
)
print("Background SHAP IV: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees: 100%|██████████| 100/100 [03:28<00:00,  2.09s/it]


cache misses: 164, cache used: 6080, M computation time: 0.54 sec, s computation time: 0.5 sec


Computing the values: 100%|██████████| 100/100 [00:02<00:00, 40.20it/s]


On Depth 6 Took: 211.6035852432251


Preprocessing the trees: 100%|██████████| 10/10 [07:07<00:00, 42.78s/it]


cache misses: 1352, cache used: 3464, M computation time: 218.14 sec, s computation time: 1.44 sec


Computing the values: 100%|██████████| 10/10 [00:06<00:00,  1.45it/s]

On Depth 9 Took: 435.13611006736755
Background SHAP IV: 211.6035852432251 & 435.13611006736755


In [17]:
# shap Package does not support Background SHAP IV

# Path Dependent SHAP

In [18]:
def pd_high_depth_woodelf_on_models_dict(
        models, consumer_data: pd.DataFrame, metric: CubeMetric, global_importance: bool = False, GPU=False
    ):
    running_times = {}
    for key, model in models.items():
        start_time = time.time()
        woodelf_for_high_depth(model, consumer_data, background_data=None, metric=metric, global_importance=global_importance)
        running_time = time.time() - start_time
        running_times[key] = running_time
        print(f"On Depth {key} Took: {running_time}")
    return running_times

def pd_simple_woodelf_on_models_dict(
        models, consumer_data: pd.DataFrame, metric: CubeMetric, global_importance: bool = False, GPU=False,
    ):
    running_times = {}
    for key, model in models.items():
        start_time = time.time()
        woodelf.simple_woodelf.calculate_path_dependent_metric(model, consumer_data, metric=metric, global_importance=global_importance)
        running_time = time.time() - start_time
        running_times[key] = running_time
        print(f"On Depth {key} Took: {running_time}")
    return running_times

def pd_shap_on_models_dict(models, consumer_data: pd.DataFrame, interaction_values: bool = False):
    running_times = {}
    for key, model in models.items():
        start_time = time.time()
        explainer = shap.TreeExplainer(model, feature_perturbation='tree_path_dependent')
        if interaction_values:
            simple_shap_values = explainer.shap_interaction_values(consumer_data)
        else:
            simple_shap_values = explainer.shap_values(consumer_data, check_additivity=False)
        running_time = time.time() - start_time
        running_times[key] = running_time
        print(f"On Depth {key} Took: {running_time}")
    return running_times

In [19]:
woodelf_running_times = pd_high_depth_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_10trees[9], 12: gbm_1trees[12], 15: gbm_1trees[15], 18: gbm_1trees[18], 21: gbm_1trees[21]},
    consumer_data=higgs_test, metric=ShapleyValues(), global_importance = False, GPU=False
)
print("Path-Dependent SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [03:15<00:00,  1.95s/it]


M time: 0.0 sec, s time: 0.88 sec (f prepare time: 0.2972373962402344)
On Depth 6 Took: 195.6161172389984


Preprocessing the trees and computing SHAP: 100%|██████████| 10/10 [03:41<00:00, 22.13s/it]


M time: 0.01 sec, s time: 0.82 sec (f prepare time: 0.27460598945617676)
On Depth 9 Took: 221.55474996566772


Preprocessing the trees and computing SHAP: 100%|██████████| 1/1 [01:55<00:00, 115.45s/it]


...

M time: 1.39 sec, s time: 0.5 sec (f prepare time: 0.15714693069458008)
On Depth 15 Took: 140.98039507865906


Preprocessing the trees and computing SHAP: 100%|██████████| 1/1 [02:40<00:00, 160.42s/it]


M time: 10.85 sec, s time: 0.52 sec (f prepare time: 0.1616358757019043)
On Depth 18 Took: 171.3589005470276


Preprocessing the trees and computing SHAP: 100%|██████████| 1/1 [02:35<00:00, 155.48s/it]

M time: 100.67 sec, s time: 0.53 sec (f prepare time: 0.16343259811401367)
On Depth 21 Took: 256.2395534515381
Path-Dependent SHAP: 195.6161172389984 & 221.55474996566772 & 115.62365174293518 & 140.98039507865906 & 171.3589005470276 & 256.2395534515381


In [20]:
woodelf_running_times = pd_simple_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_10trees[9]}, # Depth 12 crashes due to RAM (see the crashes on the fraud data)
    consumer_data=higgs_test, metric=ShapleyValues(), global_importance = False, GPU=False
)
print("Path-Dependent SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees: 100%|██████████| 100/100 [00:00<00:00, 119.34it/s]


cache misses: 164, cache used: 6080


Computing the values: 100%|██████████| 100/100 [02:37<00:00,  1.58s/it]


On Depth 6 Took: 158.90080332756042


Preprocessing the trees: 100%|██████████| 10/10 [02:44<00:00, 16.44s/it]


cache misses: 1352, cache used: 3464


Computing the values: 100%|██████████| 10/10 [02:28<00:00, 14.89s/it]

On Depth 9 Took: 313.510781288147
Path-Dependent SHAP: 158.90080332756042 & 313.510781288147


In [21]:
shap_running_times = pd_shap_on_models_dict(
    {6: gbm_10trees[6], 9: gbm_1trees[9], 12: gbm_1trees[12], 15: gbm_1trees[15], 18: gbm_1trees[18], 21: gbm_1trees[21]}, consumer_data=higgs_test,
)
print("Path-Dependent SHAP: " + " & ".join([str(shap_running_times[d]) for d in sorted(list(shap_running_times.keys()))]))

/usr/local/lib/python3.12/dist-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


On Depth 6 Took: 603.7895126342773


/usr/local/lib/python3.12/dist-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


On Depth 9 Took: 626.7133469581604


/usr/local/lib/python3.12/dist-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


...

/usr/local/lib/python3.12/dist-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


On Depth 15 Took: 384.27811789512634


/usr/local/lib/python3.12/dist-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


On Depth 18 Took: 396.7213227748871
On Depth 21 Took: 401.3533618450165
Path-Dependent SHAP: 603.7895126342773 & 626.7133469581604 & 342.5918769836426 & 384.27811789512634 & 396.7213227748871 & 401.3533618450165


/usr/local/lib/python3.12/dist-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


# Path Dependent SHAP IV

In [22]:
woodelf_running_times = pd_high_depth_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_10trees[9], 12: gbm_10trees[12], 15: gbm_10trees[15], 18: gbm_1trees[18]}, # depth 21 crashes due to RAM (see the crashes on the fraud data)
    consumer_data=higgs_test.sample(10_000, random_state=42), metric=ShapleyInteractionValues(), global_importance = False, GPU=False
)
print("Path-Dependent SHAP IV: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:04<00:00, 24.57it/s]


M time: 0.0 sec, s time: 0.73 sec (f prepare time: 0.2263350486755371)
On Depth 6 Took: 4.936939001083374


Preprocessing the trees and computing SHAP: 100%|██████████| 10/10 [00:04<00:00,  2.30it/s]


M time: 0.02 sec, s time: 0.73 sec (f prepare time: 0.21312165260314941)
On Depth 9 Took: 4.557083368301392


Preprocessing the trees and computing SHAP: 100%|██████████| 1/1 [00:02<00:00,  2.66s/it]


M time: 0.48 sec, s time: 0.43 sec (f prepare time: 0.11012578010559082)
On Depth 12 Took: 3.227536916732788


Preprocessing the trees and computing SHAP: 100%|██████████| 1/1 [00:03<00:00,  3.13s/it]


M time: 5.46 sec, s time: 0.55 sec (f prepare time: 0.12169003486633301)
On Depth 15 Took: 8.674162149429321


Preprocessing the trees and computing SHAP: 100%|██████████| 1/1 [00:03<00:00,  3.30s/it]

M time: 66.31 sec, s time: 0.62 sec (f prepare time: 0.1258091926574707)
On Depth 18 Took: 69.6998040676117
Path-Dependent SHAP IV: 4.936939001083374 & 4.557083368301392 & 3.227536916732788 & 8.674162149429321 & 69.6998040676117


In [ ]:
woodelf_running_times = pd_simple_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_10trees[9]}, # depth 12 crashes due to RAM (see the crashes on the fraud data)
    consumer_data=higgs_test.sample(10_000, random_state=42), metric=ShapleyInteractionValues(), global_importance = False, GPU=False
)
print("Path-Dependent SHAP IV: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

In [24]:
shap_running_times = pd_shap_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_10trees[9], 12: gbm_1trees[12], 15: gbm_1trees[15], 18: gbm_1trees[18], 21: gbm_1trees[21]},
    consumer_data=higgs_test.sample(10_000, random_state=42), interaction_values=True
)
print("Path-Dependent SHAP IV: " + " & ".join([str(shap_running_times[d]) for d in sorted(list(shap_running_times.keys()))]))

On Depth 6 Took: 256.83736395835876
On Depth 9 Took: 337.2024428844452
On Depth 12 Took: 191.75492548942566
On Depth 15 Took: 183.59906268119812
On Depth 18 Took: 189.20605897903442
On Depth 21 Took: 189.77042961120605
Path-Dependent SHAP IV: 256.83736395835876 & 337.2024428844452 & 191.75492548942566 & 183.59906268119812 & 189.20605897903442 & 189.77042961120605


# Crash Due to RAM

In [ ]:
woodelf_running_times = simple_woodelf_on_models_dict(
    {12: gbm_1trees[12]},
    consumer_data=higgs_test, background_data=higgs_train, metric=ShapleyValues(),
    global_importance = False, GPU=False
)
print("Background SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

In [ ]:
woodelf_running_times = simple_woodelf_on_models_dict(
    {12: gbm_1trees[12]}, # depth 15 crashes due to RAM (see the crashes on the fraud data)
    consumer_data=higgs_test.sample(10_000, random_state=42), background_data=higgs_train, metric=ShapleyInteractionValues(), global_importance = False, GPU=False
)
print("Path-Dependent SHAP IV: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees:   0%|          | 0/1 [00:00<?, ?it/s]